# Smart Property Price Predictor
Predict property values with regression models and classify properties into price bands.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## 1. Load the dataset

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()
df["Price"] = df["MedHouseVal"] * 100000
df = df.drop(columns=["MedHouseVal"])
df.head()

In [ ]:
print("Dataset shape:", df.shape)
print("\\nMissing values:\\n", df.isnull().sum())
df.describe().round(2)

## 2. Explore the data

In [ ]:
df.hist(figsize=(14, 10), bins=30)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Feature Correlation")
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
sns.scatterplot(data=df.sample(min(3000, len(df)), random_state=42), x="MedInc", y="Price", alpha=0.5)
plt.title("Median Income vs Property Price")
plt.ylabel("Property price (USD)")
plt.show()

## 3. Prepare data

In [ ]:
X = df.drop(columns="Price")
y = df["Price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

## 4. Regression models

In [ ]:
regression_models = {
    "Linear Regression": make_pipeline(StandardScaler(), LinearRegression()),
    "Polynomial Regression": make_pipeline(StandardScaler(), PolynomialFeatures(degree=2, include_bias=False), LinearRegression()),
    "Ridge Regression": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "Lasso Regression": make_pipeline(StandardScaler(), Lasso(alpha=1000, max_iter=10000))
}

regression_results = []
trained_regressors = {}

for name, model in regression_models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    trained_regressors[name] = model
    regression_results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_test, predictions)),
        "R2 Score": r2_score(y_test, predictions)
    })

regression_results = pd.DataFrame(regression_results).sort_values("RMSE")
regression_results.round(2)

In [ ]:
regression_results.set_index("Model")[["MAE", "RMSE"]].plot(kind="bar", figsize=(10, 5))
plt.ylabel("Error (USD)")
plt.title("Regression Model Error Comparison")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

regression_results.set_index("Model")["R2 Score"].plot(kind="bar", figsize=(9, 4))
plt.ylabel("R² score")
plt.title("Regression Model R² Comparison")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## 5. Classification: price bands

In [ ]:
def price_band(price):
    if price < 150000:
        return "Affordable"
    if price < 300000:
        return "Mid-range"
    return "Premium"

y_class = df["Price"].apply(price_band)
print(y_class.value_counts())

In [ ]:
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_class, test_size=0.2, random_state=42, stratify=y_class
)

classification_models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
}

classification_results = []
trained_classifiers = {}

for name, model in classification_models.items():
    model.fit(X_train_c, y_train_c)
    predictions = model.predict(X_test_c)
    trained_classifiers[name] = model
    classification_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test_c, predictions)
    })

classification_results = pd.DataFrame(classification_results).sort_values("Accuracy", ascending=False)
classification_results.round(3)

In [ ]:
best_classifier_name = classification_results.iloc[0]["Model"]
best_classifier = trained_classifiers[best_classifier_name]
class_predictions = best_classifier.predict(X_test_c)

print("Best classifier:", best_classifier_name)
print(classification_report(y_test_c, class_predictions, zero_division=0))

ConfusionMatrixDisplay.from_predictions(y_test_c, class_predictions, cmap="Blues")
plt.title(f"Confusion Matrix — {best_classifier_name}")
plt.show()

## 6. Actual vs predicted prices

In [ ]:
best_regressor_name = regression_results.iloc[0]["Model"]
best_regressor = trained_regressors[best_regressor_name]
price_predictions = best_regressor.predict(X_test)

plt.figure(figsize=(7, 5))
plt.scatter(y_test, price_predictions, alpha=0.25)
limits = [min(y_test.min(), price_predictions.min()), max(y_test.max(), price_predictions.max())]
plt.plot(limits, limits, "r--")
plt.xlabel("Actual price (USD)")
plt.ylabel("Predicted price (USD)")
plt.title(f"Actual vs Predicted — {best_regressor_name}")
plt.show()

print("Best regression model:", best_regressor_name)
print("MAE: $", round(mean_absolute_error(y_test, price_predictions), 2))
print("RMSE: $", round(np.sqrt(mean_squared_error(y_test, price_predictions)), 2))
print("R² score:", round(r2_score(y_test, price_predictions), 3))

## 7. Predict a property

Enter the property's area-level features below. This dataset represents California census-block areas, so predictions are estimates for an area rather than an individual house.

In [ ]:
sample_property = pd.DataFrame([{
    "MedInc": 5.0,
    "HouseAge": 25.0,
    "AveRooms": 5.5,
    "AveBedrms": 1.1,
    "Population": 1000.0,
    "AveOccup": 3.0,
    "Latitude": 34.05,
    "Longitude": -118.25
}])

predicted_price = best_regressor.predict(sample_property)[0]
predicted_band = best_classifier.predict(sample_property)[0]

print(f"Estimated property-area value: ${predicted_price:,.0f}")
print("Predicted price band:", predicted_band)
sample_property

## 8. Predict using custom values

In [ ]:
def predict_property(med_income, house_age, average_rooms, average_bedrooms,
                     population, average_occupancy, latitude, longitude):
    property_data = pd.DataFrame([{
        "MedInc": med_income,
        "HouseAge": house_age,
        "AveRooms": average_rooms,
        "AveBedrms": average_bedrooms,
        "Population": population,
        "AveOccup": average_occupancy,
        "Latitude": latitude,
        "Longitude": longitude
    }])
    estimated_price = best_regressor.predict(property_data)[0]
    predicted_category = best_classifier.predict(property_data)[0]
    return {
        "Estimated value (USD)": round(float(estimated_price), 2),
        "Predicted price band": predicted_category
    }

predict_property(5.0, 25, 5.5, 1.1, 1000, 3.0, 34.05, -118.25)

## Conclusion
The project compares Linear, Polynomial, Ridge, and Lasso Regression for estimating property-area values, then compares Logistic Regression and KNN for price-band classification. The best models are selected using test-set metrics. Results depend on the dataset and should be treated as estimates, not official property valuations.